# SonicForge — Development Sandbox

Use this notebook to quickly test libraries, API calls, and ideas **without touching the main codebase**.

Run it with the project's virtual environment kernel:
```
source ../.venv/bin/activate
jupyter notebook
```

## 1. Read FLAC Metadata with mutagen

In [ ]:
from mutagen.flac import FLAC

# Replace with any local FLAC file path
path = "/path/to/your/file.flac"

audio = FLAC(path)
for key, val in audio.items():
    print(f"{key:20} = {val}")

print(f"\nPictures: {len(audio.pictures)}")

## 2. Fetch Artwork from iTunes Search API

In [ ]:
import requests
import urllib.parse
from IPython.display import Image, display

artist = "Adele"
album  = "19"

term  = urllib.parse.quote_plus(f"{artist} {album}")
url   = f"https://itunes.apple.com/search?term={term}&entity=album&limit=5"
data  = requests.get(url, timeout=5).json()

for r in data.get('results', []):
    thumb = r.get('artworkUrl100', '')
    print(r.get('collectionName'), '->', thumb)
    if thumb:
        display(Image(url=thumb))

## 3. Fetch Artwork from Deezer API

In [ ]:
import requests
from IPython.display import Image, display

artist = "Adele"
album  = "19"

url  = f'https://api.deezer.com/search/album?q=artist:"{artist}" album:"{album}"&limit=5'
data = requests.get(url, timeout=5).json()

for r in data.get('data', []):
    cover = r.get('cover_medium', '')
    print(r.get('title'), '->', cover)
    if cover:
        display(Image(url=cover))

## 4. Image Processing — Crop & Resize to 500×500 JPEG

In [ ]:
import io, requests
from PIL import Image as PILImage
from IPython.display import Image, display

img_url = "https://is1-ssl.mzstatic.com/image/thumb/Music/v4/3b/4c/f1/3b4cf100x100bb.jpg"

raw = requests.get(img_url, timeout=5).content
img = PILImage.open(io.BytesIO(raw))

# Centre-crop to square
w, h = img.size
m = min(w, h)
img = img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2))
img = img.resize((500, 500), PILImage.Resampling.LANCZOS)
if img.mode != 'RGB':
    img = img.convert('RGB')

buf = io.BytesIO()
img.save(buf, format='JPEG')
display(Image(data=buf.getvalue()))
print(f"Output size: {len(buf.getvalue())} bytes")

## 5. Write Metadata Back to a FLAC File

In [ ]:
from mutagen.flac import FLAC

# ⚠️  This WILL modify the file on disk!
path = "/path/to/your/file.flac"

audio = FLAC(path)
audio['artist'] = 'New Artist'
audio['album']  = 'New Album'
audio.save()

print("Tags saved.")